# SuryaOCR Evaluation on Synthetic Handwritten Data

This notebook evaluates SuryaOCR performance on synthetic handwritten German nursing notes.

**Metrics Calculated:**
- Accuracy (sequence similarity)
- CER (Character Error Rate)
- WER (Word Error Rate)
- NED (Normalized Edit Distance)
- Vocabulary Precision (unique word overlap)
- Vocabulary Recall (unique word overlap)
- Vocabulary F1-Score (unique word overlap)

**Output:**
- Individual results: `results/metrics/surya/individual/`
- Summary CSV: `results/metrics/surya/summary.csv`
- Summary JSON: `results/metrics/surya/summary.json`
- Detailed results JSON: `results/metrics/surya/detailed_results.json`


## 1. Setup and Imports


In [1]:
import sys
import os
from pathlib import Path

# Add project root to path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

from PIL import Image
from surya.foundation import FoundationPredictor
from surya.recognition import RecognitionPredictor
from surya.detection import DetectionPredictor

import json
import pandas as pd
from tqdm.auto import tqdm
from datetime import datetime

# Import our custom metrics module
from src.evaluation.metrics import calculate_all_metrics, format_metrics_display

print("✓ All imports successful")
print(f"Project root: {project_root}")


/home/bashar/bachelor-thesis-nursing-agents/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✓ All imports successful
Project root: /home/bashar/bachelor-thesis-nursing-agents


## 2. Configuration


In [2]:
# Paths
IMAGES_DIR = project_root / "data" / "synthetic" / "output" / "images"
LABELS_DIR = project_root / "data" / "synthetic" / "output" / "labels"
RESULTS_DIR = project_root / "results" / "metrics" / "surya"
INDIVIDUAL_RESULTS_DIR = RESULTS_DIR / "individual"

# Create output directories
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
INDIVIDUAL_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# ==================================================================================
# EVALUATION STRATEGY - Choose one:
# ==================================================================================
# Option A: Use fixed test set (RECOMMENDED for fair comparison)
USE_TEST_SET = True
TEST_SET_FILE = project_root / "data" / "test_set_600.json"  # or test_set_500.json, test_set_complete.json

# Option B: Random sampling (NOT recommended for comparing OCR methods)
# USE_TEST_SET = False
# LIMIT_SAMPLES = None  # Set to a number (e.g., 10) for testing, None for all images
# ==================================================================================

print(f"Images directory: {IMAGES_DIR}")
print(f"Labels directory: {LABELS_DIR}")
print(f"Results directory: {RESULTS_DIR}")

if USE_TEST_SET:
    if TEST_SET_FILE.exists():
        print(f"✓ Using FIXED TEST SET: {TEST_SET_FILE}")
        print("  (Ensures fair comparison across OCR methods)")
    else:
        print(f"⚠ TEST SET FILE NOT FOUND: {TEST_SET_FILE}")
        print("  Run: python scripts/create_test_set.py --strategy balanced-600 --output data/test_set_600.json")
        USE_TEST_SET = False
        LIMIT_SAMPLES = None
else:
    print(f"⚠ Using random sampling: {LIMIT_SAMPLES if 'LIMIT_SAMPLES' in globals() and LIMIT_SAMPLES else 'All images'}")
    print("  Note: This may not be fair for comparing different OCR methods!")


Images directory: /home/bashar/bachelor-thesis-nursing-agents/data/synthetic/output/images
Labels directory: /home/bashar/bachelor-thesis-nursing-agents/data/synthetic/output/labels
Results directory: /home/bashar/bachelor-thesis-nursing-agents/results/metrics/surya
✓ Using FIXED TEST SET: /home/bashar/bachelor-thesis-nursing-agents/data/test_set_600.json
  (Ensures fair comparison across OCR methods)


## 3. Initialize Surya OCR


In [3]:
print("Initializing Surya OCR predictors...")
print("This may take a moment on first run (downloading ~1 GB foundation model)...")

foundation_predictor = FoundationPredictor()
recognition_predictor = RecognitionPredictor(foundation_predictor)
detection_predictor = DetectionPredictor()

print("✓ Surya OCR predictors initialized successfully")


Initializing Surya OCR predictors...
This may take a moment on first run (downloading ~1 GB foundation model)...
✓ Surya OCR predictors initialized successfully


## 4. Load Image and Label Pairs


In [4]:
# Load image-label pairs based on evaluation strategy
valid_pairs = []

if USE_TEST_SET and TEST_SET_FILE.exists():
    # Load from fixed test set
    print(f"Loading test set from: {TEST_SET_FILE}")
    with open(TEST_SET_FILE, 'r', encoding='utf-8') as f:
        test_set_data = json.load(f)

    print(f"Test set contains {test_set_data['test_set_size']} samples")

    for sample in test_set_data['samples']:
        img_path = project_root / sample['image_path']
        label_path = project_root / sample['label_path']

        if img_path.exists() and label_path.exists():
            valid_pairs.append((img_path, label_path))
        else:
            print(f"⚠ Missing files for {sample['sample_id']}")

    print(f"✓ Loaded {len(valid_pairs)} samples from FIXED TEST SET")
    print(f"  (Same samples will be used for all OCR methods)")

else:
    # Fall back to all images or limited sampling
    print("Loading all images from directory...")
    image_files = sorted(list(IMAGES_DIR.glob("*.png")))

    if 'LIMIT_SAMPLES' in globals() and LIMIT_SAMPLES:
        image_files = image_files[:LIMIT_SAMPLES]

    print(f"Found {len(image_files)} images")

    # Verify corresponding label files exist
    for img_path in image_files:
        label_path = LABELS_DIR / f"{img_path.stem}.json"
        if label_path.exists():
            valid_pairs.append((img_path, label_path))
        else:
            print(f"⚠ Warning: No label file found for {img_path.name}")

    print(f"✓ {len(valid_pairs)} valid image-label pairs ready")
    if not USE_TEST_SET:
        print(f"⚠ WARNING: Not using fixed test set - may not be fair for OCR comparison!")


Loading test set from: /home/bashar/bachelor-thesis-nursing-agents/data/test_set_600.json
Test set contains 559 samples
✓ Loaded 559 samples from FIXED TEST SET
  (Same samples will be used for all OCR methods)


## 5. Batch Processing with OCR and Evaluation


In [5]:
results_list = []
failed_samples = []

print(f"\n{'=' * 80}")
print("STARTING BATCH OCR PROCESSING")
print(f"{'=' * 80}\n")

for img_path, label_path in tqdm(valid_pairs, desc="Processing images"):
    sample_id = img_path.stem

    try:
        # Load ground truth
        with open(label_path, 'r', encoding='utf-8') as f:
            label_data = json.load(f)

        expected_text = label_data.get('full_text', '')

        # Run Surya OCR
        image = Image.open(img_path)
        predictions = recognition_predictor([image], det_predictor=detection_predictor)

        # Extract text from OCR results (join all detected text lines)
        text_lines = predictions[0].text_lines
        predicted_text = " ".join([line.text for line in text_lines])

        # Calculate all metrics
        metrics = calculate_all_metrics(expected_text, predicted_text)

        # Prepare result dictionary
        result = {
            'sample_id': sample_id,
            'image_path': str(img_path.relative_to(project_root)),
            'expected_text': expected_text,
            'predicted_text': predicted_text,
            'expected_length': len(expected_text),
            'predicted_length': len(predicted_text),
            'num_detections': len(text_lines),
            **metrics,  # Add all metrics
            'detections': [
                {
                    'text': line.text,
                    'confidence': float(line.confidence),
                    'polygon': line.polygon
                }
                for line in text_lines
            ]
        }

        results_list.append(result)

        # Save individual result
        individual_result_path = INDIVIDUAL_RESULTS_DIR / f"{sample_id}_result.json"
        with open(individual_result_path, 'w', encoding='utf-8') as f:
            json.dump(result, f, ensure_ascii=False, indent=2)

    except Exception as e:
        print(f"✗ Error processing {sample_id}: {str(e)}")
        failed_samples.append({'sample_id': sample_id, 'error': str(e)})

print(f"\n{'=' * 80}")
print(f"✓ Successfully processed: {len(results_list)} samples")
print(f"✗ Failed: {len(failed_samples)} samples")
print(f"{'=' * 80}")



STARTING BATCH OCR PROCESSING



Processing images:   0%|          | 0/559 [00:00<?, ?it/s]

Processing images: 100%|██████████| 559/559 [21:21<00:00,  2.29s/it]


✓ Successfully processed: 559 samples
✗ Failed: 0 samples


## 6. Create Summary Statistics


In [6]:
if len(results_list) > 0:
    # Create DataFrame for analysis
    df_results = pd.DataFrame([
        {
            'sample_id': r['sample_id'],
            'accuracy': r['accuracy'],
            'cer': r['cer'],
            'wer': r['wer'],
            'ned': r['ned'],
            'vocab_precision': r['vocab_precision'],
            'vocab_recall': r['vocab_recall'],
            'vocab_f1': r['vocab_f1'],
            'expected_length': r['expected_length'],
            'predicted_length': r['predicted_length'],
            'num_detections': r['num_detections']
        }
        for r in results_list
    ])

    # Calculate summary statistics
    summary_stats = {
        'method': 'SuryaOCR',
        'model_version': '0.17.1',
        'num_samples': len(results_list),
        'timestamp': datetime.now().isoformat(),
        'metrics': {
            'accuracy': {
                'mean': float(df_results['accuracy'].mean()),
                'std': float(df_results['accuracy'].std()),
                'min': float(df_results['accuracy'].min()),
                'max': float(df_results['accuracy'].max()),
                'median': float(df_results['accuracy'].median())
            },
            'cer': {
                'mean': float(df_results['cer'].mean()),
                'std': float(df_results['cer'].std()),
                'min': float(df_results['cer'].min()),
                'max': float(df_results['cer'].max()),
                'median': float(df_results['cer'].median())
            },
            'wer': {
                'mean': float(df_results['wer'].mean()),
                'std': float(df_results['wer'].std()),
                'min': float(df_results['wer'].min()),
                'max': float(df_results['wer'].max()),
                'median': float(df_results['wer'].median())
            },
            'ned': {
                'mean': float(df_results['ned'].mean()),
                'std': float(df_results['ned'].std()),
                'min': float(df_results['ned'].min()),
                'max': float(df_results['ned'].max()),
                'median': float(df_results['ned'].median())
            },
            'vocab_precision': {
                'mean': float(df_results['vocab_precision'].mean()),
                'std': float(df_results['vocab_precision'].std()),
                'min': float(df_results['vocab_precision'].min()),
                'max': float(df_results['vocab_precision'].max()),
                'median': float(df_results['vocab_precision'].median())
            },
            'vocab_recall': {
                'mean': float(df_results['vocab_recall'].mean()),
                'std': float(df_results['vocab_recall'].std()),
                'min': float(df_results['vocab_recall'].min()),
                'max': float(df_results['vocab_recall'].max()),
                'median': float(df_results['vocab_recall'].median())
            },
            'vocab_f1': {
                'mean': float(df_results['vocab_f1'].mean()),
                'std': float(df_results['vocab_f1'].std()),
                'min': float(df_results['vocab_f1'].min()),
                'max': float(df_results['vocab_f1'].max()),
                'median': float(df_results['vocab_f1'].median())
            }
        }
    }

    print("Summary Statistics:")
    print("=" * 80)
    for metric_name, stats in summary_stats['metrics'].items():
        print(f"\n{metric_name.upper()}:")
        print(f"  Mean:   {stats['mean']:.2f}%")
        print(f"  Median: {stats['median']:.2f}%")
        print(f"  Std:    {stats['std']:.2f}%")
        print(f"  Range:  [{stats['min']:.2f}%, {stats['max']:.2f}%]")
    print("=" * 80)
else:
    print("⚠ No results to summarize")


Summary Statistics:

ACCURACY:
  Mean:   62.96%
  Median: 72.58%
  Std:    34.86%
  Range:  [0.93%, 100.00%]

CER:
  Mean:   14.43%
  Median: 5.05%
  Std:    23.79%
  Range:  [0.00%, 292.78%]

WER:
  Mean:   43.72%
  Median: 27.27%
  Std:    69.39%
  Range:  [0.00%, 1037.25%]

NED:
  Mean:   13.15%
  Median: 5.04%
  Std:    16.97%
  Range:  [0.00%, 75.62%]

VOCAB_PRECISION:
  Mean:   65.61%
  Median: 75.00%
  Std:    30.01%
  Range:  [2.44%, 100.00%]

VOCAB_RECALL:
  Mean:   66.16%
  Median: 75.47%
  Std:    29.71%
  Range:  [1.89%, 100.00%]

VOCAB_F1:
  Mean:   65.85%
  Median: 75.11%
  Std:    29.83%
  Range:  [2.13%, 100.00%]


## 7. Save Results


In [7]:
if len(results_list) > 0:
    # Save CSV (for easy analysis in Excel/Python)
    csv_path = RESULTS_DIR / "summary.csv"
    df_results.to_csv(csv_path, index=False, encoding='utf-8')
    print(f"✓ Saved CSV summary: {csv_path}")

    # Save summary JSON
    summary_json_path = RESULTS_DIR / "summary.json"
    with open(summary_json_path, 'w', encoding='utf-8') as f:
        json.dump(summary_stats, f, ensure_ascii=False, indent=2)
    print(f"✓ Saved JSON summary: {summary_json_path}")

    # Save detailed results (all data including OCR detections)
    detailed_json_path = RESULTS_DIR / "detailed_results.json"
    with open(detailed_json_path, 'w', encoding='utf-8') as f:
        json.dump(results_list, f, ensure_ascii=False, indent=2)
    print(f"✓ Saved detailed results: {detailed_json_path}")

    # Save failed samples if any
    if failed_samples:
        failed_path = RESULTS_DIR / "failed_samples.json"
        with open(failed_path, 'w', encoding='utf-8') as f:
            json.dump(failed_samples, f, ensure_ascii=False, indent=2)
        print(f"⚠ Saved failed samples log: {failed_path}")

    print(f"\n{'=' * 80}")
    print("ALL RESULTS SAVED SUCCESSFULLY")
    print(f"{'=' * 80}")
    print(f"\nResults location: {RESULTS_DIR}")
    print(f"  - summary.csv ({len(df_results)} rows)")
    print(f"  - summary.json (aggregate statistics)")
    print(f"  - detailed_results.json (full OCR output)")
    print(f"  - individual/ ({len(results_list)} files)")
else:
    print("⚠ No results to save")


✓ Saved CSV summary: /home/bashar/bachelor-thesis-nursing-agents/results/metrics/surya/summary.csv
✓ Saved JSON summary: /home/bashar/bachelor-thesis-nursing-agents/results/metrics/surya/summary.json
✓ Saved detailed results: /home/bashar/bachelor-thesis-nursing-agents/results/metrics/surya/detailed_results.json

ALL RESULTS SAVED SUCCESSFULLY

Results location: /home/bashar/bachelor-thesis-nursing-agents/results/metrics/surya
  - summary.csv (559 rows)
  - summary.json (aggregate statistics)
  - detailed_results.json (full OCR output)
  - individual/ (559 files)


## 8. Quick Analysis and Visualization


In [8]:
if len(results_list) > 0:
    # Display top 5 best and worst performing samples
    print("\n" + "=" * 80)
    print("TOP 5 BEST PERFORMING SAMPLES (by Vocabulary F1-Score)")
    print("=" * 80)
    top_5 = df_results.nlargest(5, 'vocab_f1')
    print(top_5[['sample_id', 'accuracy', 'cer', 'wer', 'vocab_f1']].to_string(index=False))

    print("\n" + "=" * 80)
    print("TOP 5 WORST PERFORMING SAMPLES (by Vocabulary F1-Score)")
    print("=" * 80)
    bottom_5 = df_results.nsmallest(5, 'vocab_f1')
    print(bottom_5[['sample_id', 'accuracy', 'cer', 'wer', 'vocab_f1']].to_string(index=False))

    # Basic statistics
    print("\n" + "=" * 80)
    print("BASIC STATISTICS")
    print("=" * 80)
    print(f"Samples with accuracy > 90%: {(df_results['accuracy'] > 90).sum()} ({(df_results['accuracy'] > 90).sum() / len(df_results) * 100:.1f}%)")
    print(f"Samples with accuracy > 80%: {(df_results['accuracy'] > 80).sum()} ({(df_results['accuracy'] > 80).sum() / len(df_results) * 100:.1f}%)")
    print(f"Samples with accuracy > 70%: {(df_results['accuracy'] > 70).sum()} ({(df_results['accuracy'] > 70).sum() / len(df_results) * 100:.1f}%)")
    print(f"Samples with CER < 10%: {(df_results['cer'] < 10).sum()} ({(df_results['cer'] < 10).sum() / len(df_results) * 100:.1f}%)")
    print(f"Samples with Vocabulary F1 > 80%: {(df_results['vocab_f1'] > 80).sum()} ({(df_results['vocab_f1'] > 80).sum() / len(df_results) * 100:.1f}%)")
else:
    print("⚠ No data for analysis")



TOP 5 BEST PERFORMING SAMPLES (by Vocabulary F1-Score)
  sample_id  accuracy  cer  wer  vocab_f1
sample_0086     100.0  0.0  0.0     100.0
sample_1384     100.0  0.0  0.0     100.0
sample_2335     100.0  0.0  0.0     100.0
sample_1206     100.0  0.0  0.0     100.0
sample_1113     100.0  0.0  0.0     100.0

TOP 5 WORST PERFORMING SAMPLES (by Vocabulary F1-Score)
  sample_id  accuracy       cer        wer  vocab_f1
sample_2224  3.353659 71.162791 100.000000  2.127660
sample_0019 20.866142 47.368421 100.000000  2.985075
sample_0219  3.921569 61.065574 106.349206  3.305785
sample_1517  6.565657 60.496614 108.196721  3.389831
sample_0998  7.098765 54.624277 112.727273  4.000000

BASIC STATISTICS
Samples with accuracy > 90%: 217 (38.8%)
Samples with accuracy > 80%: 258 (46.2%)
Samples with accuracy > 70%: 290 (51.9%)
Samples with CER < 10%: 362 (64.8%)
Samples with Vocabulary F1 > 80%: 244 (43.6%)


## 9. Example: Detailed View of a Sample

Run this cell to see detailed results for a specific sample:


In [9]:
# Change this to any sample_id you want to inspect
SAMPLE_TO_INSPECT = results_list[0]['sample_id'] if len(results_list) > 0 else None

if SAMPLE_TO_INSPECT:
    sample_result = next((r for r in results_list if r['sample_id'] == SAMPLE_TO_INSPECT), None)

    if sample_result:
        print("=" * 80)
        print(f"DETAILED RESULTS FOR: {SAMPLE_TO_INSPECT}")
        print("=" * 80)

        print(f"\n📄 Expected Text ({sample_result['expected_length']} chars):")
        print("-" * 80)
        print(sample_result['expected_text'])

        print(f"\n🤖 Predicted Text ({sample_result['predicted_length']} chars):")
        print("-" * 80)
        print(sample_result['predicted_text'])

        print(f"\n{format_metrics_display(sample_result)}")

        print(f"\n{'=' * 80}")
        print(f"DETECTED TEXT REGIONS ({sample_result['num_detections']} detections)")
        print("=" * 80)
        for i, detection in enumerate(sample_result['detections'], 1):
            print(f"\n{i}. '{detection['text']}'")
            print(f"   Confidence: {detection['confidence']:.2f}")
else:
    print("⚠ No samples available for inspection")


DETAILED RESULTS FOR: sample_0246

📄 Expected Text (1069 chars):
--------------------------------------------------------------------------------
Patient: Albert Schuster Datum: 29.10.2025 29.10.2025, Hr. Schuster, 92 J., Zi. 32, PG 5
KOMPLETTE IMMOBILITÄT, bettlägerig seit Schlaganfall 2024. Grundpflege 7:00 Uhr: -
Ganzwaschwäsche im Bett, Pat. kann nicht mithelfen - Inkontinenzversorgung (Windel, Bettschutz)
- Katheter suprapubisch, Urin klar-gelb, ca. 800ml/24h - Mundpflege intensiv Lagerungsplan:
7:00 rechts - 10:00 links - 13:00 rechts - 16:00 links - 19:00 rechts Anti-Dekubitus-Matratze
im Einsatz. DEKUBITUS-STATUS: - Sakral Grad 2, ca. 3x4 cm, Wundversorgung mit HydroClean
- Beide Fersen freigelagert mit Gelkissen - Haut ansonsten intakt Ernährung: - Vollkost püriert
über Löffel - Frühstück: 5 Löffel Brei - Mittag: ca. 10 Löffel Gemüsebrei, 3 Löffel Kartoffelpüree -
Trinkmenge: 600ml (in kleinen Schlucken) Kommunikation: Pat. reagiert mit Augenkontakt, kann
nicht sprechen. Verst

---
## ℹ️ Usage Instructions

### Quick Start (Step by Step)
1. **Run Cell 1**: Install dependencies
2. **Run Cell 3**: Initialize imports and verify paths
3. **Run Cell 5**: Configure paths and settings (optional: set `USE_TEST_SET = False` and `LIMIT_SAMPLES = 10` for quick testing)
4. **Run Cell 7**: Initialize Surya OCR predictors (takes ~30 seconds first time, downloads ~1 GB model)
5. **Run Cell 9**: Load and validate image-label pairs
6. **Run Cell 11**: **Main processing** — Process all images with OCR (this may take a while!)
7. **Run Cell 13**: Calculate summary statistics
8. **Run Cell 15**: Save all results
9. **Run Cells 17 & 19**: View quick analysis and sample details

### Expected Output Files
- `results/metrics/surya/summary.csv` — All samples with metrics
- `results/metrics/surya/summary.json` — Aggregate statistics
- `results/metrics/surya/detailed_results.json` — Full OCR output
- `results/metrics/surya/individual/` — Per-sample JSON files

### Testing First
Set `USE_TEST_SET = False` and `LIMIT_SAMPLES = 10` in Cell 5 to test on a small subset before processing all images.

### Note on Model Memory
SuryaOCR loads a large foundation model into GPU memory. Ensure you have at least 8 GB VRAM available.
